# ITERATORS

---

## Iterables and Iterators

### Definitions

An object is said to be iterable if you can loop over the object. An iterable object produces an **iterator** object when it needs to be looped over.

**iterator:** an iterator is a special object that actually performs the traversing. It remembers the current position and knows how to get the next item, to traverse the sequence.

To make a object **iterable**, it must follow python's iterable protocol - i.e., it must define an `__iter__()` method, that returns an iterator object that can traverse the object.

Also, an **iterator** object needs to define 2 methods:
- `__iter__()`: this should return `self` cuz python expects iterators themselves to be iterable
- `__next__()`: should return the next value of the **iterable** object, and raise `StopIteration()` exception when the sequence terminates.

Consider the following example:

In [1]:
my_list = [1, 2, 3]

it = iter(my_list)  # iter(<iterable object>) returns an iterator object 
print(f"iterator object: {it}")

print(next(it))
print(next(it))

iterator object: <list_iterator object at 0x00000173C32132B0>
1
2


**Note:** Python expects an iterator objects to themselves be iterable. Thus:
```python
iter(it) is it
```

Now, this allows:
```python
it = iter(some_iterator)
```
to work properly.

> Hence, _all iterators are iterable, but not all iterables are iterators._

### Working of `for` loop

Consider the following loop:

In [2]:
for x in my_list:
    print(x)

1
2
3


This is _exactly_ equivalent to:

In [4]:
_iter = iter(my_list)       # calls my_list.__iter__()

while True:
    try:
        x = next(_iter)      # calls _iter.__next__()
        print(x)
    except StopIteration:
        break

1
2
3


Python calls `iter()` method of the _iterable_ object once, to get the iterator object. Then, python uses `next(iterator)`, whick calls `iterator.__next__()`, to get the values from the _iterable_ object.

Therefore, if an object properly implements `__iter__` and `__next__`, the following python functionalities automatically works:
- `for` loop
- `list()`
- `sum()`
- `in` operator
- tuple unpacking

---

## Implementing Iterators and Iterables

### Iterable + Iterator as the same object:

In [9]:
class CountUp:
    def __init__(self, start, stop):
        self.stop = stop
        self.current = start
    
    def __iter__(self):
        return self     # must be true for all iterators
    
    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

In [10]:
for n in CountUp(1, 4):
    print(n)

1
2
3


> _The `__iter__` method retuns `self`. This is called iterator/iterable unification. Here, the same object serves the role of iterable as well as iterator._

### Separation Pattern: Iterable + Iterator as 2 classes

In [11]:
c = CountUp(1, 4)

list_1 = list(c)
print(list_1)

list_2 = list(c)
print(list_2)

[1, 2, 3]
[]


In the above code, there's a clear problem. We can loop over one `CountUp` instance only once. After exhaustion, `self.current` is permenantly past `self.current`. Thus, calling `iter()` again returns the same exhausted object.

The clean solution used by Python's built-in types is to separtate the iterator from the iterable object. The iterable is then *stateless* and *reusable*, while the iterator holds the *mutable* state.

In [16]:
class CountRange():
    def __init__(self, start, stop):
        self.start = start
        self.stop = stop

    def __iter__(self):
        return CountRangeIterator(self.start, self.stop) 
    

class CountRangeIterator():
    def __init__(self, start, stop):
        self.stop = stop
        self.current = start
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration()
        value = self.current
        self.current += 1
        return value

In [17]:
r = CountRange(1, 4)

list_1 = list(r)
print(list_1)

list_2 = list(r)
print(list_2)

print(list_1 is list_2)

[1, 2, 3]
[1, 2, 3]
False


This is exactly what python's `list` does. The list itself is stateless, but the `iter(my_list)` returns a fresh `list_iterator` object each time, with it's own internal indexing starting at 0. Hence, multiple iterators over the same list are all *independent*.

---

## Miscellenous Concepts

### Infinte Iterators

Iterators generate values on demand, so there's no requirment that they ever stop. Thus, we can implement an infinte iterator, that never raises `StopIteration()`, essntially functioning as an infinte iterator.

In [18]:
class NaturalNumbers:
    def __init__(self):
        self.current = 0

    def __iter__(self):
        return self
    
    def __next__(self):
        self.current += 1
        return self.current

In [19]:
nats = NaturalNumbers()
print(next(nats))
print(next(nats))
print(next(nats))

1
2
3


### Two-Argument Form of `iter()`

`iter()` has a powerful 2nd form:
```python
iter(callable, sentinel)
```

This creates an iterator that calls `callable()` repeatedly, and stops as soon as the return value equals `sentinel`.

For e.g., consider the following example:

```python
with open("data.bin", "rb") as f:
    for chunk in iter(lambda: f.read(4096), b""):
        process(chumk)
```

`f.read(4096)` returns `b""` when EOF is reached - That's the sentinel. Python wraps this into a clean iterator with no explicit `while True` or manual logic break

### Propagation of `StopIteration`

`StopIteration` is a real exception that travels up the call stack normally - untill something catches it. Python's built-in mechanisms, such as the `for` loop, `list()`, `sum()`, `next()`, etc. catches this exception

But if `StopIteration`escapes out of a generator (raised inside a `yield`-based function unintentionally), python converts it to a `RuntimeError` since Python 3.7. This prevents bugs where a nested iterator's exhaustion silently terminates an outer generator.

```python
def gen():
    yield from gen2()
    yield 1
```

if `gen2()` raises a `StopIteration` exception here, it's not handled in `gen()`. This could terminate the `gen()` as well, instead of yielding `1`.

**RULE:** Inside a generator, signal exhaustion by simply returning using `return`, not by manually raising `StopIteration`.

### Generators are Iterators

Generators are also iterators, which are automatically built from a function body. When writing a generator, python automatically generates the `__iter__` and `__next__` methods under the hood, using positions of `yield` statements as pause points.

In [23]:
def count_up(n):
    for i in range(n):
        yield i

# Proof - a generator object has both __iter__ and __next__ methods:
g = count_up(4)
print(g)

print(g.__iter__())     # should return g itself
print(g.__next__())     # should print 0

<generator object count_up at 0x00000173C38AF510>
<generator object count_up at 0x00000173C38AF510>
0


Generators are the most ergonomic way to write an iterator as it allows us to express the same idea, but as a linear code.

On the other hand, writing a class with `__iter__` and `__next__` is more powerful because it can have:
- multiple methods
- richer state
- cleaner API
However, this is more verbose and may not be suitable for small tasks.

### Iterator Power Tools: `itertools`

The `itertools` module is built entirely on this protocol, providing composable, lazy iterator building blocks:

| Function              | What it does                           |
| --------------------- | -------------------------------------- |
| `count(start)`        | Infinite counter: 0, 1, 2, ...         |
| `islice(it, n)`       | Take first `n` items from any iterator |
| `chain(a, b)`         | Flatten multiple iterables into one    |
| `zip_longest(a, b)`   | Like `zip` but pads shorter iterable   |
| `takewhile(pred, it)` | Yield while predicate is True          |
| `groupby(it, key)`    | Group consecutive elements by key      |
| `cycle(it)`           | Repeat an iterable infinitely          |

---

## Three-Layer View of Python Iteration

Iteration in python can be thought of as 3 different layers, built on top of each other:

**Layer 1 — The data source (iterable).** Knows how to create an iterator. Stateless and reusable. A list, file, dict, or your own class with `__iter__`.

**Layer 2 — The cursor (iterator).** Holds position. Has `__next__` to advance. Stateful and single-use. Created fresh by `iter()` at the start of each loop.

**Layer 3 — The consumer.** A `for` loop, `list()`, `sum()`, `zip()` — anything that calls `next()` repeatedly until `StopIteration`. Completely generic — works on _any_ iterator without knowing what's inside it.

The power of this design is that **Layer 3 is completely decoupled from Layer 1**. A `for` loop doesn't know or care whether it's walking a list, reading a file line by line, fetching database rows, or computing prime numbers. As long as Layer 2 (the iterator) fulfills the protocol, the consumer works. This is why you can swap a list for a generator in most code with zero changes to the consuming side — they speak the same language.